In [4]:
import pandas as pd
import numpy as np
import random
import math

# ==========================================
# 1. The Data and Preprocessing
# ==========================================
def preprocess_data(file_path, target_date='1/4/2020', target_line='429A', target_hour=8):
    """
    Loads and preprocesses the Istanbul Hourly Public Transport dataset.
    Extracts the Grand Total Hourly Demand for a specific date, line, and hour.
    """
    print(f"Loading data from {file_path}...")
    df = pd.read_csv(file_path, encoding='ISO-8859-9')
    
    # Filter for the single date
    # Adjust `target_date` string format to match your CSV's exact format (e.g., '1/4/2020' or '2020-01-04')
    # Filter for the line and the specific morning rush hour
    df_filtered = df[(df['transition_date'] == target_date) & 
                     (df['line'] == target_line) & 
                     (df['transition_hour'] == target_hour)]
    
    if df_filtered.empty:
        print("Warning: No data found for the given filters. Proceeding with a dummy demand for demonstration.")
        return 600  # Return a dummy demand of 600 passengers for demonstration
    
    # Aggregate the demand: Sum the number_of_passenger across all matching rows
    grand_total_hourly_demand = df_filtered['number_of_passenger'].sum()
    print(f"Grand Total Hourly Demand for line {target_line} on {target_date} at {target_hour}:00 is {grand_total_hourly_demand}")
    return grand_total_hourly_demand

# ==========================================
# 2. The Simulation & Constants
# ==========================================

# Problem constraints
VEHICLE_CAPACITY = 50
FLEET_SIZE = 15
TOTAL_MINUTES = 60
INTERVAL_MINUTES = 5

def simulate_hour(schedule, total_hourly_demand):
    """
    Evaluates a proposed timetable (chromosome) by simulating the queue of passengers.
    Returns the number of stranded passengers at the end of the hour (fitness score).
    """
    # Divide demand evenly across the twelve 5-minute intervals
    demand_per_interval = total_hourly_demand / (TOTAL_MINUTES / INTERVAL_MINUTES)
    
    queue = 0
    # Create a frequency map of bus arrival minutes
    bus_arrivals = {m: 0 for m in range(TOTAL_MINUTES)}
    for minute in schedule:
        # Safeguard to ensure genes are always valid minutes
        if 0 <= minute < TOTAL_MINUTES:
            bus_arrivals[minute] += 1
            
    # Simulate the hour minute by minute
    for minute in range(TOTAL_MINUTES):
        # Passengers arrive steadily every 5 minutes and form a queue over time
        if minute % INTERVAL_MINUTES == 0:
            queue += demand_per_interval
            
        # If a scheduled bus arrives, it removes up to 50 passengers from the queue
        buses_here = bus_arrivals[minute]
        for _ in range(buses_here):
            queue = max(0, queue - VEHICLE_CAPACITY)
            
    return queue

# ==========================================
# 3. The Algorithms
# ==========================================

# --- Phase 1: Genetic Algorithm (GA) ---

def create_individual():
    """Integer Encoding: Chromosome is a list of integers representing departure minutes."""
    return [random.randint(0, TOTAL_MINUTES - 1) for _ in range(FLEET_SIZE)]

def crossover(parent1, parent2):
    """Single-point crossover to combine schedules."""
    pt = random.randint(1, FLEET_SIZE - 2)
    child1 = parent1[:pt] + parent2[pt:]
    child2 = parent2[:pt] + parent1[pt:]
    return child1, child2

def mutate(individual, mutation_rate=0.1):
    """Randomly change departure minutes with a given probability."""
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            individual[i] = random.randint(0, TOTAL_MINUTES - 1)
    return individual

def run_genetic_algorithm(total_hourly_demand, pop_size=100, generations=50, mutation_rate=0.2):
    """Executes Phase 1: The Genetic Algorithm to explore the search space."""
    population = [create_individual() for _ in range(pop_size)]
    
    best_overall = None
    best_fitness = float('inf')
    
    for gen in range(generations):
        # Objective: minimize stranded passengers
        fitness_scores = [(ind, simulate_hour(ind, total_hourly_demand)) for ind in population]
        fitness_scores.sort(key=lambda x: x[1])
        
        # Track best solution found so far
        if fitness_scores[0][1] < best_fitness:
            best_overall = fitness_scores[0][0]
            best_fitness = fitness_scores[0][1]
            
        # Elitism: Automatically carry over the top 10%
        next_gen = [ind for ind, _ in fitness_scores[:int(pop_size * 0.1)]]
        
        # Selection & Reproduction
        while len(next_gen) < pop_size:
            # Tournament selection (size = 3)
            tournament = random.sample(fitness_scores, 3)
            tournament.sort(key=lambda x: x[1])
            parent1 = tournament[0][0]
            
            tournament = random.sample(fitness_scores, 3)
            tournament.sort(key=lambda x: x[1])
            parent2 = tournament[0][0]
            
            child1, child2 = crossover(parent1, parent2)
            next_gen.append(mutate(child1, mutation_rate))
            if len(next_gen) < pop_size:
                next_gen.append(mutate(child2, mutation_rate))
                
        population = next_gen
        
    return best_overall, best_fitness


# --- Phase 2: Simulated Annealing (SA) via Sequential Refinement ---

def scramble_mutation(schedule):
    """Swapping or shifting departure minutes for SA neighborhood generation."""
    neighbor = schedule.copy()
    if random.random() < 0.5:
        # Swap two random genes
        idx1, idx2 = random.sample(range(FLEET_SIZE), 2)
        neighbor[idx1], neighbor[idx2] = neighbor[idx2], neighbor[idx1]
    else:
        # Shift a random gene's minute slightly
        idx = random.randint(0, FLEET_SIZE - 1)
        shift = random.randint(-5, 5)
        neighbor[idx] = max(0, min(TOTAL_MINUTES - 1, neighbor[idx] + shift))
    return neighbor

def run_simulated_annealing(initial_schedule, total_hourly_demand, initial_temp=100.0, cooling_rate=0.95, iterations=1000):
    """Executes Phase 2: SA algorithm for local refinement of the best GA solution."""
    current_schedule = initial_schedule
    current_fitness = simulate_hour(current_schedule, total_hourly_demand)
    
    best_schedule = current_schedule.copy()
    best_fitness = current_fitness
    
    temp = initial_temp
    
    for i in range(iterations):
        # Generate a neighboring schedule using Scramble Mutation
        neighbor = scramble_mutation(current_schedule)
        neighbor_fitness = simulate_hour(neighbor, total_hourly_demand)
        
        # Calculate change in fitness (delta cost)
        delta = neighbor_fitness - current_fitness
        
        # Acceptance logic
        if delta < 0:
            # Better solution: accept immediately
            current_schedule = neighbor
            current_fitness = neighbor_fitness
            
            # Keep track of absolute best
            if current_fitness < best_fitness:
                best_schedule = current_schedule.copy()
                best_fitness = current_fitness
        else:
            # Worse solution: Temperature-controlled Boltzmann probability to escape local optima
            if temp > 0.01:
                probability = math.exp(-delta / temp)
                if random.random() < probability:
                    current_schedule = neighbor
                    current_fitness = neighbor_fitness
                    
        # Cooling down the temperature
        temp *= cooling_rate
        
    return best_schedule, best_fitness


# ==========================================
# Main Execution Block
# ==========================================
if __name__ == "__main__":
    # Provide the exact path to the dataset
    file_path = "hourly_transportation_202001.csv"
    
    # 1. Preprocessing
    # Ensure the date format accurately matches the dataset ('1/4/2020' or '2020-01-04')
    # Update target_date, target_line, and target_hour to test different slices of data
    demand = preprocess_data(file_path, target_date='1/4/2020', target_line='429A', target_hour=8)
    
    print("\n--- Phase 1: Genetic Algorithm ---")
    best_ga_schedule, ga_fitness = run_genetic_algorithm(demand, pop_size=50, generations=50)
    print(f"Best GA Schedule (unsorted minutes): {best_ga_schedule}")
    print(f"Best GA Fitness (stranded passengers): {ga_fitness:.2f}")
    
    print("\n--- Phase 2: Simulated Annealing (Sequential Refinement) ---")
    best_sa_schedule, sa_fitness = run_simulated_annealing(best_ga_schedule, demand, iterations=500)
    
    # Sorting the final schedule makes it easier to read chronologically
    print(f"Final SA Refined Schedule (sorted minutes): {sorted(best_sa_schedule)}")
    print(f"Final SA Fitness (stranded passengers): {sa_fitness:.2f}")
    
    print("\n--- Pipeline Complete ---")


Loading data from hourly_transportation_202001.csv...

--- Phase 1: Genetic Algorithm ---
Best GA Schedule (unsorted minutes): [13, 3, 57, 30, 57, 52, 59, 37, 30, 23, 13, 20, 34, 56, 34]
Best GA Fitness (stranded passengers): 0.00

--- Phase 2: Simulated Annealing (Sequential Refinement) ---
Final SA Refined Schedule (sorted minutes): [3, 13, 13, 20, 23, 30, 30, 34, 34, 37, 52, 56, 57, 57, 59]
Final SA Fitness (stranded passengers): 0.00

--- Pipeline Complete ---
